# Enhanced RAG + GPT-3.5+ GPT-4

## Imports

In [ ]:
# --- Imports ---
# Standard Libraries
import os
import time
from typing import Tuple, List

# Data Manipulation
import pandas as pd
import numpy as np

# NLP Libraries
import spacy
import nltk  # For potential text processing
from transformers import (
    AutoTokenizer,
    AutoModel,
)  # For tokenizer and general transformer models

# Langchain
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import ChatOpenAI  # Assuming this is for GPT-3.5
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone as PineconeVectorStore
from langchain.prompts import PromptTemplate  # If you plan to use prompt templates

# Pinecone
from pinecone import Pinecone, ServerlessSpec

# Data Loading and Manipulation
from datasets import load_dataset, Dataset

# Retrieval
from rank_bm25 import BM25Okapi

# Web Requests
import requests

# Evaluation
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    context_entity_recall,
    answer_similarity,
    answer_correctness,
)  # Assuming these are the correct Ragas metrics
from ragas import evaluate

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Other
from scipy.stats import entropy

## Constants

### API Keys

In [ ]:
DEEP_INFRA_API_KEY = "SET_YOUR_API_KEY"
os.environ["OPENAI_API_KEY"] = "SET_YOUR_API_KEY"

### DeepInfra

In [ ]:
DEEP_INFRA_API_BASE = "https://api.deepinfra.com/v1/openai"

## Initialization

In [ ]:
# Initialize Spacy for Named Entity Recognition (NER)
nlp = spacy.load("en_core_web_sm")
# Initialize Tokenizer (GPT-2 is a safe default)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

## Helper Functions

In [ ]:
# --- Helper Functions ---
def count_tokens(text: str) -> int:
    """
    Calculates the number of tokens in a given text string.

    Args:
        text: The text string to tokenize.

    Returns:
        The number of tokens in the text.
    """
    return len(tokenizer.encode(text))

## Wikidata

In [ ]:
def query_wikidata(entity: str) -> List[str]:
    """
    Queries Wikidata for information about a given entity.

    Args:
        entity: The entity to query (e.g., "Albert Einstein").

    Returns:
        A list of strings containing information about the entity, or an empty list
        if no information is found.
    """

    url = "https://query.wikidata.org/sparql"
    query = f"""
    SELECT ?item ?itemLabel WHERE {{
        ?item rdfs:label "{entity}"@en.
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 3
    """
    headers = {"Accept": "application/json"}

    try:
        response = requests.get(url, params={"query": query}, headers=headers)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        data = response.json()
        results = data.get("results", {}).get("bindings", [])
        return [result["itemLabel"]["value"] for result in results]
    except requests.exceptions.RequestException as e:
        print(f"Error querying Wikidata: {e}")
        return []


def enrich_text_with_wikidata(text: str) -> str:
    """
    Identifies named entities in the given text and retrieves corresponding
    information from Wikidata.

    Args:
        text: The text to enrich.

    Returns:
        A string containing Wikidata information about the entities found in the text.
        Returns an empty string if no relevant entities are found.
    """

    doc = nlp(text)
    entities = [
        ent.text
        for ent in doc.ents
        if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT"]
    ]  # Filter for relevant entity types

    wikidata_knowledge = []
    for entity in set(entities):  # Process each unique entity
        results = query_wikidata(entity)
        if results:
            wikidata_knowledge.append(f"{entity}: {'; '.join(results)}")

    return "\n".join(wikidata_knowledge) if wikidata_knowledge else ""

## Dataset

### Corpus

In [ ]:
dataset = load_dataset("jamescalam/llama-2-arxiv-papers-chunked", split="train")
data = dataset.to_pandas()
corpus = [x["chunk"] for _, x in data.iterrows()]

### Evaluation Dataset

In [ ]:
eval_dataset = load_dataset("hanyueshf/ml-arxiv-papers-qa", split="train")
eval_df = pd.DataFrame(eval_dataset)
eval_df = eval_df.remove_columns(["id", "context"])
eval_df = eval_df.rename_column("answer", "ground_truth")

In [ ]:
eval_df_limited = eval_df

## Model Embedding

In [ ]:
embed_model = OpenAIEmbeddings(
    model="text-embedding-ada-002", openai_api_key=os.environ["OPENAI_API_KEY"]
)

## Okapi BM25 (Best Matching 25)

In [ ]:
def initialize_bm25_index(corpus: List[str]) -> BM25Okapi:
    """
    Initializes a BM25Okapi index for sparse retrieval.

    Args:
        corpus: A list of text documents to index.

    Returns:
        A BM25Okapi index.
    """
    tokenized_corpus = [doc.split() for doc in corpus]
    return BM25Okapi(tokenized_corpus)

### Initialization

In [ ]:
bm25_index = initialize_bm25_index(corpus)

## Pinecone

### Function

In [ ]:
def initialize_pinecone(
    api_key: str, environment: str, index_name: str, embed_dimension: int
) -> Pinecone.Index:
    """
    Initializes the Pinecone index for vector storage.

    Returns:
        An instance of the Pinecone index ready for use.
    """
    pc = Pinecone(api_key=api_key)

    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=embed_dimension,
            metric="cosine",  # Or 'euclidean' based on your methodology
            spec=ServerlessSpec(cloud="aws", region=environment),
        )

    return pc.Index(index_name)

### Environment

In [ ]:
PINECONE_API_KEY = "SET_YOUR_API_KEY"
PINECONE_ENVIRONMENT = "us-east-1"
PINECONE_INDEX_NAME = "dynamic-rag-hybrid-retrieval"
PINECONE_EMBED_DIMENSION = 1536

### Initialization

In [ ]:
index = initialize_pinecone(
    PINECONE_API_KEY,
    PINECONE_ENVIRONMENT,
    PINECONE_INDEX_NAME,
    embed_dimension=PINECONE_EMBED_DIMENSION,
)

## Retrieval

In [ ]:
def retrieve_dense(index: Pinecone.Index, query: str, top_k: int) -> List[dict]:
    """
    Performs dense retrieval using Pinecone.

    Args:
        index: The Pinecone index to query.
        query: The query string.
        top_k: The number of results to retrieve.

    Returns:
        A list of dictionaries, where each dictionary represents a retrieved document
        and its associated score and metadata.
    """

    xq = embed_model.embed_query(query)  # Embed the query
    results = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return results.get("matches", [])

In [ ]:
def retrieve_sparse(
    bm25_index: BM25Okapi, query: str, corpus: List[str], top_k: int
) -> List[Tuple[int, float]]:
    """
    Performs sparse retrieval using BM25.

    Args:
        bm25_index: The BM25Okapi index.
        query: The query string.
        corpus: The list of documents.
        top_k: The number of top documents to return.

    Returns:
        A list of tuples, where each tuple contains the document index and its BM25 score.
    """
    tokenized_query = query.split()
    doc_scores = bm25_index.get_scores(tokenized_query)
    # Get indices of the top_k documents
    top_indices = np.argsort(doc_scores)[::-1][:top_k]
    return [(i, doc_scores[i]) for i in top_indices]

In [ ]:
def retrieve_hybrid(
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    top_k: int,
    alpha: float = 0.5,  # Weight for dense retrieval
    dynamic_threshold: bool = True,
) -> Tuple[List[dict], float]:
    """
    Performs hybrid retrieval, combining dense and sparse retrieval.

    Args:
        index: The Pinecone index for dense retrieval.
        bm25_index: The BM25Okapi index for sparse retrieval.
        corpus: The list of documents.
        query: The query string.
        top_k: The number of results to retrieve from each method.
        alpha: Weighting factor for dense retrieval (1-alpha for sparse).
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A tuple containing:
            - A list of retrieved documents, sorted by the combined score.
            - The dynamic threshold (if used) or the fixed threshold.
    """

    dense_results = retrieve_dense(index, query, top_k)
    sparse_results = retrieve_sparse(bm25_index, query, corpus, top_k)

    # Align results by document index (assuming the index IDs allow this)
    combined_results = []
    for dense_hit in dense_results:
        doc_id = int(dense_hit["id"].split("-")[-1])  # Extract original doc ID
        dense_score = dense_hit["score"]
        sparse_score = next(
            (score for index, score in sparse_results if index == doc_id), 0
        )  # Default to 0 if not found
        combined_score = alpha * dense_score + (1 - alpha) * sparse_score
        combined_results.append(
            {
                "doc_id": dense_hit["id"],
                "text": dense_hit["metadata"]["text"],
                "dense_score": dense_score,
                "sparse_score": sparse_score,
                "combined_score": combined_score,
            }
        )

    combined_results.sort(key=lambda x: x["combined_score"], reverse=True)

    threshold = 0.4  # Default threshold
    if dynamic_threshold:
        scores = [hit["combined_score"] for hit in combined_results]
        entropy_val = entropy(scores) if len(scores) > 1 else 0
        threshold = max(0.4, 0.6 - (0.2 * entropy_val))

    filtered_results = [
        hit["text"] for hit in combined_results if hit["combined_score"] >= threshold
    ]
    return filtered_results, threshold

## Prompt

In [ ]:
def create_rag_prompt(query: str, context: str, wikidata_info: str) -> str:
    """
    Creates a prompt for the LLM, incorporating retrieved context and Wikidata information.

    Args:
        query: The user's query.
        context: The retrieved context from the knowledge base.
        wikidata_info: Information retrieved from Wikidata.

    Returns:
        A formatted prompt string.
    """

    prompt = f"""Using the context below, answer the query.
    Context={context}
    Additional Knowledge={wikidata_info}
    Query={query}
    If there is no context, the context is not clear or the context and the question are not related,
    answer as GPT starting with 'As there is no context provided, I will answer as a GPT'."""
    return prompt

In [ ]:
def create_rag_prompt(query: str, context: str, wikidata_info: str) -> str:
    """
    Creates a prompt for the LLM, incorporating retrieved context and Wikidata information.

    Args:
        query: The user's query.
        context: The retrieved context from the knowledge base.
        wikidata_info: Information retrieved from Wikidata.

    Returns:
        A formatted prompt string.
    """

    prompt = f"""Using the context below, answer the query.
    Context={context}
    Additional Knowledge={wikidata_info}
    Query={query}
    If there is no context, the context is not clear or the context and the question are not related,
    answer as GPT starting with 'As there is no context provided, I will answer naturally'."""
    return prompt

In [ ]:
def generate_response(llm, prompt: str) -> str:
    """
    Generates a response from the LLM given a prompt.

    Args:
        llm: The language model object (e.g., ChatOpenAI).
        prompt: The prompt to send to the LLM.

    Returns:
        The generated response string.
    """

    messages = [HumanMessage(content=prompt)]
    response = llm(messages)
    return response.content

## Evaluation Pipeline

### Enhanced RAG Pipeline

In [ ]:
# --- Main RAG Function ---
def advanced_rag_pipeline(
    llm,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    retrieval_method: str = "hybrid",
    top_k: int = 10,
    alpha: float = 0.5,
    use_wikidata: bool = True,
    dynamic_threshold: bool = True,
) -> Tuple[str, dict]:
    """
    Performs Retrieval Augmented Generation (RAG) with hybrid retrieval,
    dynamic thresholding, and optional Wikidata integration.

    Args:
        llm: The language model to use for response generation.
        index: The Pinecone index for dense retrieval.
        bm25_index: The BM25Okapi index for sparse retrieval.
        corpus: The list of documents.
        query: The user's query.
        retrieval_method: The retrieval method to use ("dense", "sparse", or "hybrid").
        top_k: The number of documents to retrieve.
        alpha: Weight for dense retrieval in hybrid mode.
        use_wikidata: Whether to enrich context with Wikidata information.
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A tuple containing:
            - The generated response string.
            - A dictionary of metadata (e.g., retrieval scores, threshold).
    """

    start_time = time.time()

    # 1. Retrieval
    if retrieval_method == "dense":
        retrieved_context = [
            hit["metadata"]["text"] for hit in retrieve_dense(index, query, top_k)
        ]
        threshold = None  # Not applicable for dense-only
    elif retrieval_method == "sparse":
        retrieved_context = [
            corpus[index]
            for index, score in retrieve_sparse(bm25_index, query, corpus, top_k)
        ]
        threshold = None  # Not applicable for sparse-only
    elif retrieval_method == "hybrid":
        retrieved_context, threshold = retrieve_hybrid(
            index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold
        )
    else:
        raise ValueError(f"Invalid retrieval method: {retrieval_method}")

    context = "\n".join(retrieved_context) if retrieved_context else "No context found."

    # 2. Wikidata Enrichment
    wikidata_info = (
        enrich_text_with_wikidata(context)
        if use_wikidata and context != "No context found."
        else ""
    )

    # 3. Prompting and Generation
    prompt = create_rag_prompt(query, context, wikidata_info)
    response = generate_response(llm, prompt)

    end_time = time.time()

    metadata = {
        "retrieval_method": retrieval_method,
        "retrieved_docs": len(retrieved_context),
        "wikidata_entities": len(wikidata_info.split("\n")) if wikidata_info else 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": threshold,
    }

    return response, metadata

### Build Evaluation Dataset

In [ ]:
# --- Evaluation ---
def build_evaluation_dataset(
    llm,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    dataframe: pd.DataFrame,
    retrieval_method: str,
    top_k: int,
    alpha: float,
    use_wikidata: bool,
    dynamic_threshold: bool,
    max_questions: int = 200,  # Limit for testing
) -> pd.DataFrame:
    """
    Builds a dataset for evaluating the RAG pipeline.

    Args:
        llm: The language model to use.
        index: The Pinecone index.
        bm25_index: The BM25 index.
        corpus: The list of documents.
        dataframe: A Pandas DataFrame containing questions and ground truth answers.
        retrieval_method: The retrieval method to use ("dense", "sparse", or "hybrid").
        top_k: The number of documents to retrieve.
        alpha: Weight for dense retrieval in hybrid mode.
        use_wikidata: Whether to use Wikidata.
        dynamic_threshold: Whether to use dynamic thresholding.
        max_questions: The maximum number of questions to process.

    Returns:
        A Pandas DataFrame with added 'answer' and 'contexts' columns.
    """

    answers = []
    contexts = []
    questions = dataframe["question"].tolist()[
        :max_questions
    ]  # Limit questions for efficiency

    for question in tqdm(questions, desc=f"Evaluating with {retrieval_method}"):
        try:
            response, metadata = advanced_rag_pipeline(
                llm=llm,
                index=index,
                bm25_index=bm25_index,
                corpus=corpus,
                query=question,
                retrieval_method=retrieval_method,
                top_k=top_k,
                alpha=alpha,
                use_wikidata=use_wikidata,
                dynamic_threshold=dynamic_threshold,
            )
            answers.append(response)
            contexts.append(metadata)  # Store metadata as context
        except Exception as e:
            print(f"Error processing question: {question}. Error: {e}")
            answers.append(None)
            contexts.append(None)

    eval_df = dataframe[: len(answers)].copy()  # Keep the original data aligned
    eval_df["answer"] = answers
    eval_df["contexts"] = contexts  # Store metadata
    return eval_df

### RAGAS (Retrieval Augmented Generation Assessment)

In [ ]:
def prepare_ragas_dataset(eval_df: pd.DataFrame) -> Dataset:
    """
    Prepares a Pandas DataFrame for evaluation with the Ragas library.

    Args:
        eval_df: A Pandas DataFrame containing 'question', 'ground_truth', 'answer',
                 and 'contexts' columns.

    Returns:
        A Hugging Face Dataset formatted for Ragas.
    """

    ragas_df = eval_df[["question", "ground_truth", "answer", "contexts"]].copy()

    # Ensure 'contexts' is a list of strings (Ragas requirement)
    def ensure_list_of_strings(example):
        if not isinstance(example["contexts"], list):
            example["contexts"] = [str(example["contexts"])]
        return example

    ragas_dataset = Dataset.from_pandas(ragas_df).map(ensure_list_of_strings)
    return ragas_dataset

## GPT-4 (gpt-4)

In [ ]:
chat_gpt_4 = ChatOpenAI(model_name="gpt-4", openai_api_key=os.environ["OPENAI_API_KEY"])

## GPT-3.5 (gpt-3.5-turbo)

### Initialization

In [ ]:
chat_gpt_3 = ChatOpenAI(
    model_name="gpt-3.5-turbo", openai_api_key=os.environ["OPENAI_API_KEY"]
)

### Evaluation

In [ ]:
# 3. Evaluation

chat_gpt_3_llm_config = {
    "llm": chat_gpt_3,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

eval_dense_df = build_evaluation_dataset(
    **chat_gpt_3_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)
eval_hybrid_df = build_evaluation_dataset(
    **chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)
eval_hybrid_no_wikidata_df = build_evaluation_dataset(
    **chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

eval_hybrid_fixed_threshold_df = build_evaluation_dataset(
    **chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

ragas_dataset_dense = prepare_ragas_dataset(eval_dense_df)
ragas_dataset_hybrid = prepare_ragas_dataset(eval_hybrid_df)
ragas_dataset_hybrid_no_wikidata = prepare_ragas_dataset(eval_hybrid_no_wikidata_df)
ragas_dataset_hybrid_fixed_threshold = prepare_ragas_dataset(
    eval_hybrid_fixed_threshold_df
)

# Evaluate with Ragas
result_dense = evaluate(
    dataset=ragas_dataset_dense,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,  # Handle potential errors gracefully
)
result_hybrid = evaluate(
    dataset=ragas_dataset_hybrid,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
result_hybrid_no_wikidata = evaluate(
    dataset=ragas_dataset_hybrid_no_wikidata,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
result_hybrid_fixed_threshold = evaluate(
    dataset=ragas_dataset_hybrid_fixed_threshold,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)

print("\n--- Ragas Evaluation Results ---")
print("\nDense Only:")
print(result_dense.to_pandas().mean(axis=0))

print("\nHybrid:")
print(result_hybrid.to_pandas().mean(axis=0))

print("\nHybrid No Wikidata:")
print(result_hybrid_no_wikidata.to_pandas().mean(axis=0))

print("\nHybrid Fixed Threshold:")
print(result_hybrid_fixed_threshold.to_pandas().mean(axis=0))

## Mistral (mistralai/Mistral-7B-Instruct-v0.1)

### Initialization

In [ ]:
chat_gpt_3 = ChatOpenAI(
    model_name="gpt-3.5-turbo", openai_api_key=os.environ["OPENAI_API_KEY"]
)
chat_gpt_4 = ChatOpenAI(model_name="gpt-4", openai_api_key=os.environ["OPENAI_API_KEY"])

mistral_model = "mistralai/Mistral-7B-Instruct-v0.1"

mistral_llm = ChatOpenAI(
    model_name=mistral_model,
    openai_api_key=DEEP_INFRA_API_KEY,
    openai_api_base=DEEP_INFRA_API_BASE,
)

### Evaluation

In [ ]:
# 3. Evaluation with Mistral

mistral_llm_config = {
    "llm": mistral_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

eval_dense_mistral_df = build_evaluation_dataset(
    **mistral_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)
eval_hybrid_mistral_df = build_evaluation_dataset(
    **mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)
eval_hybrid_no_wikidata_mistral_df = build_evaluation_dataset(
    **mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)
eval_hybrid_fixed_threshold_mistral_df = build_evaluation_dataset(
    **mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

# 4. Evaluate with Ragas
ragas_dataset_dense_mistral = prepare_ragas_dataset(eval_dense_mistral_df)
ragas_dataset_hybrid_mistral = prepare_ragas_dataset(eval_hybrid_mistral_df)
ragas_dataset_hybrid_no_wikidata_mistral = prepare_ragas_dataset(
    eval_hybrid_no_wikidata_mistral_df
)
ragas_dataset_hybrid_fixed_threshold_mistral = prepare_ragas_dataset(
    eval_hybrid_fixed_threshold_mistral_df
)

result_dense_mistral = evaluate(
    dataset=ragas_dataset_dense_mistral,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
result_hybrid_mistral = evaluate(
    dataset=ragas_dataset_hybrid_mistral,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
result_hybrid_no_wikidata_mistral = evaluate(
    dataset=ragas_dataset_hybrid_no_wikidata_mistral,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
result_hybrid_fixed_threshold_mistral = evaluate(
    dataset=ragas_dataset_hybrid_fixed_threshold_mistral,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)

print("\n--- Ragas Evaluation Results (Mistral) ---")
print("\nDense Only (Mistral):")
print(result_dense_mistral.to_pandas().mean(axis=0))

print("\nHybrid (Mistral):")
print(result_hybrid_mistral.to_pandas().mean(axis=0))

print("\nHybrid No Wikidata (Mistral):")
print(result_hybrid_no_wikidata_mistral.to_pandas().mean(axis=0))

print("\nHybrid Fixed Threshold (Mistral):")
print(result_hybrid_fixed_threshold_mistral.to_pandas().mean(axis=0))

## Llama3 (meta-llama/Llama-3.3-70B-Instruct-Turbo) + Llama4 (meta-llama/Llama-4-Scout-17B-16E-Instruct) + Deepseek (deepseek-ai/DeepSeek-V3)

In [ ]:
!pip install together

In [ ]:
# --- Together AI Integration (Llama 3, Llama 4, DeepSeek) ---
from together import Together

# Ensure you have your Together AI API key set as an environment variable
os.environ["TOGETHER_API_KEY"] = "SET_YOUR_API_KEY"  # Replace with your actual key
together_client = Together()

In [ ]:
def generate_with_together(model_name: str, prompt: str, max_tokens: int = 200) -> str:
    """
    Generates text using a model hosted by Together AI.

    Args:
        model_name: The name of the Together AI model to use (e.g., "meta-llama/Llama-3.3-70B-Instruct-Turbo").
        prompt: The prompt to generate text from.
        max_tokens: The maximum number of tokens to generate.

    Returns:
        The generated text.
    """
    try:
        response = together_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error generating with Together AI ({model_name}): {e}")
        return "Sorry, I cannot provide a response to this query."

In [ ]:
# --- Modified RAG Pipelines for Together AI Models ---


def rag_pipeline_dense_only_together(
    model_name: str,
    index: Pinecone.Index,
    query: str,
    top_k: int = 10,
    use_wikidata: bool = True,
) -> Tuple[str, dict]:
    """
    RAG pipeline using only dense retrieval with a Together AI hosted LLM.
    """
    start_time = time.time()
    retrieved_context = [
        hit["metadata"]["text"] for hit in retrieve_dense(index, query, top_k)
    ]
    context = "\n".join(retrieved_context) if retrieved_context else "No context found."
    wikidata_info = (
        enrich_text_with_wikidata(context)
        if use_wikidata and context != "No context found."
        else ""
    )

    prompt = create_rag_prompt(query, context, wikidata_info)
    response = generate_with_together(model_name, prompt)

    end_time = time.time()
    metadata = {
        "retrieval_method": f"dense_{model_name.split('/')[-1].lower().replace('-', '_')}",
        "retrieved_docs": len(retrieved_context),
        "wikidata_entities": len(wikidata_info.split("\n")) if wikidata_info else 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": None,
    }
    return response, metadata


def rag_pipeline_hybrid_no_wikidata_together(
    model_name: str,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    top_k: int = 10,
    alpha: float = 0.5,
    dynamic_threshold: bool = True,
) -> Tuple[str, dict]:
    """
    RAG pipeline using hybrid retrieval without Wikidata with a Together AI hosted LLM.
    """
    start_time = time.time()
    retrieved_context, threshold = retrieve_hybrid(
        index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold
    )
    context = "\n".join(retrieved_context) if retrieved_context else "No context found."
    prompt = create_rag_prompt(query, context, "")  # Empty wikidata_info
    response = generate_with_together(model_name, prompt)

    end_time = time.time()
    metadata = {
        "retrieval_method": f"hybrid_no_wikidata_{model_name.split('/')[-1].lower().replace('-', '_')}",
        "retrieved_docs": len(retrieved_context),
        "wikidata_entities": 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": threshold,
    }
    return response, metadata


def rag_pipeline_hybrid_wikidata_fixed_threshold_together(
    model_name: str,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    top_k: int = 10,
    alpha: float = 0.5,
    fixed_threshold: float = 0.6,
    use_wikidata: bool = True,
) -> Tuple[str, dict]:
    """
    RAG pipeline using hybrid retrieval, Wikidata, and a fixed threshold with a Together AI hosted LLM.
    """
    start_time = time.time()
    retrieved_context, _ = retrieve_hybrid(
        index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold=False
    )
    context = "\n".join(retrieved_context) if retrieved_context else "No context found."
    wikidata_info = (
        enrich_text_with_wikidata(context)
        if use_wikidata and context != "No context found."
        else ""
    )

    hybrid_scores = []
    for hit in retrieve_hybrid(
        index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold=False
    )[0]:
        hybrid_scores.append(hit["combined_score"])
    filtered_context = [
        c
        for i, c in enumerate(retrieved_context)
        if hybrid_scores[i] >= fixed_threshold
    ]
    context = "\n".join(filtered_context) if filtered_context else "No context found."

    prompt = create_rag_prompt(query, context, wikidata_info)
    response = generate_with_together(model_name, prompt)

    end_time = time.time()
    metadata = {
        "retrieval_method": f"hybrid_wikidata_fixed_threshold_{model_name.split('/')[-1].lower().replace('-', '_')}",
        "retrieved_docs": len(filtered_context),
        "wikidata_entities": len(wikidata_info.split("\n")) if wikidata_info else 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": fixed_threshold,
    }
    return response, metadata


def advanced_rag_pipeline_together(
    model_name: str,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    retrieval_method: str = "hybrid",
    top_k: int = 10,
    alpha: float = 0.5,
    use_wikidata: bool = True,
    dynamic_threshold: bool = True,
) -> Tuple[str, dict]:
    """
    Performs Retrieval Augmented Generation (RAG) with hybrid retrieval,
    dynamic thresholding, and optional Wikidata integration using a Together AI hosted LLM.
    """
    start_time = time.time()

    if retrieval_method == "dense":
        retrieved_context = [
            hit["metadata"]["text"] for hit in retrieve_dense(index, query, top_k)
        ]
        threshold = None
    elif retrieval_method == "sparse":
        retrieved_context = [
            corpus[index]
            for index, score in retrieve_sparse(bm25_index, query, corpus, top_k)
        ]
        threshold = None
    elif retrieval_method == "hybrid":
        retrieved_context, threshold = retrieve_hybrid(
            index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold
        )
    else:
        raise ValueError(f"Invalid retrieval method: {retrieval_method}")

    context = "\n".join(retrieved_context) if retrieved_context else "No context found."
    wikidata_info = (
        enrich_text_with_wikidata(context)
        if use_wikidata and context != "No context found."
        else ""
    )

    prompt = create_rag_prompt(query, context, wikidata_info)
    response = generate_with_together(model_name, prompt)

    end_time = time.time()
    metadata = {
        "retrieval_method": f"{retrieval_method}_{model_name.split('/')[-1].lower().replace('-', '_')}",
        "retrieved_docs": len(retrieved_context),
        "wikidata_entities": len(wikidata_info.split("\n")) if wikidata_info else 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": threshold,
    }
    return response, metadata

In [ ]:
def build_evaluation_dataset_together(
    model_name: str,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    dataframe: pd.DataFrame,
    retrieval_method: str,
    top_k: int,
    alpha: float,
    use_wikidata: bool,
    dynamic_threshold: bool,
    max_questions: int = 200,
) -> pd.DataFrame:
    """
    Builds a dataset for evaluating the RAG pipeline with a Together AI hosted LLM.
    """
    answers = []
    contexts = []
    questions = dataframe["question"].tolist()[:max_questions]

    for question in tqdm(
        questions, desc=f"Evaluating with {retrieval_method} ({model_name})"
    ):
        try:
            response, metadata = advanced_rag_pipeline_together(
                model_name=model_name,
                index=index,
                bm25_index=bm25_index,
                corpus=corpus,
                query=question,
                retrieval_method=retrieval_method,
                top_k=top_k,
                alpha=alpha,
                use_wikidata=use_wikidata,
                dynamic_threshold=dynamic_threshold,
            )
            answers.append(response)
            contexts.append(metadata)
        except Exception as e:
            print(f"Error processing question: {question}. Error: {e}")
            answers.append(None)
            contexts.append(None)

    eval_df = dataframe[: len(answers)].copy()
    eval_df["answer"] = answers
    eval_df["contexts"] = contexts
    return eval_df

In [ ]:
llama_3_model = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
llama_4_model = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
deepseek_model = "deepseek-ai/DeepSeek-V3"

### Llama 3

In [ ]:
# Evaluate with Llama 3

llama_3_llm_config = {
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

eval_llama3_df = build_evaluation_dataset_together(
    **llama_3_llm_config,
    model_name=llama_3_model,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
    max_questions=5,  # Even smaller for multiple LLMs
)
ragas_dataset_llama3 = prepare_ragas_dataset(eval_llama3_df)
result_llama3 = evaluate(
    dataset=ragas_dataset_llama3,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
print(f"\n--- Ragas Evaluation Results (Llama 3) ---")
print(result_llama3.to_pandas().mean(axis=0))

### Llama 4

In [ ]:
# Evaluate with Llama 4

llama_4_llm_config = {
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

eval_llama4_df = build_evaluation_dataset_together(
    **llama_4_llm_config,
    model_name=llama_4_model,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
    max_questions=5,
)
ragas_dataset_llama4 = prepare_ragas_dataset(eval_llama4_df)
result_llama4 = evaluate(
    dataset=ragas_dataset_llama4,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
print(f"\n--- Ragas Evaluation Results (Llama 4) ---")
print(result_llama4.to_pandas().mean(axis=0))

### Deepseek

In [ ]:
# Evaluate with DeepSeek

deepseek_llm_config = {
    "llm": deepseek_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

eval_deepseek_df = build_evaluation_dataset_together(
    **deepseek_llm_config,
    model_name=deepseek_model,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
    max_questions=5,
)
ragas_dataset_deepseek = prepare_ragas_dataset(eval_deepseek_df)
result_deepseek = evaluate(
    dataset=ragas_dataset_deepseek,
    metrics=[answer_relevancy, answer_similarity, answer_correctness],
    raise_exceptions=False,
)
print(f"\n--- Ragas Evaluation Results (DeepSeek) ---")
print(result_deepseek.to_pandas().mean(axis=0))